### Settings KeplerGl

In [190]:
from keplergl import KeplerGl

config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": 47.3769,
            "longitude": 8.47,
            "zoom": 10,
            "bearing": 0,
            "pitch": 0
        },
        "visState": {
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "Trajectories": [
                            {"name": "length (km)"}
                        ]
                    },
                    "enabled": True
                }
            }
        }
    }
}

map_1 = KeplerGl(height=700, config=config)

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


### Open JSON

In [191]:
import json

# Load full trajectory linestrings from JSON
json_path = '../WorldMove/output/trajectories_lines.json'
with open(json_path, 'r') as f:
    trajectories = json.load(f)

### Calculate Drivingdistance in km

In [ ]:
import math
import json
import os
import datetime


def haversine(p0, p1):
    """Calculate geodesic distance between two lon/lat points in meters."""
    lon1, lat1 = p0
    lon2, lat2 = p1
    r = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(min(1, math.sqrt(a)))


def compute_feature_length(feature):
    """Compute the length of a LineString feature using haversine distance, returns km."""
    geometry = feature.get('geometry', {})
    if geometry.get('type') != 'LineString':
        return None
    coords = geometry.get('coordinates', [])
    total = 0.0
    for p0, p1 in zip(coords, coords[1:]):
        total += haversine(p0, p1)
    return round(total / 1000, 2)  # Return in kilometers


# Compute lengths for all features in the loaded trajectories
print(f"Computing lengths for {len(trajectories.get('features', []))} features...")
lengths = []
for feature in trajectories.get('features', []):
    length_km = compute_feature_length(feature)
    feature.setdefault('properties', {})['computed_length_km'] = length_km
    lengths.append(length_km)

# Print summary
valid_lengths = [l for l in lengths if l is not None]
print(f"\nComputation complete!")
print(f"  Features processed: {len(lengths)}")
print(f"  LineStrings with lengths: {len(valid_lengths)}")
if valid_lengths:
    print(f"  min length (km): {min(valid_lengths):.2f}")
    print(f"  max length (km): {max(valid_lengths):.2f}")
    print(f"  mean length (km): {sum(valid_lengths) / len(valid_lengths):.2f}")

length_20 = [l for l in valid_lengths if l < 20]
if length_20:
    print(f"  LineStrings < 20km: {len(length_20)}")

# Create a subset dataset for features shorter than 20km
short_features = [
    feature
    for feature in trajectories.get('features', [])
    if feature.get('properties', {}).get('computed_length_km') is not None
    and feature['properties']['computed_length_km'] < 20
]
short_trajectories = {
    key: trajectories[key]
    for key in ('type', 'name', 'crs')
    if key in trajectories
}
short_trajectories['features'] = short_features

# Add generation timestamp to the metadata
calc_time = datetime.datetime.utcnow().isoformat() + 'Z'
short_trajectories['generated_at'] = calc_time

# Save filtered trajectories to timestamped JSON file and a _latest copy
output_dir = os.path.join(os.getcwd(), '..', 'WorldMove', 'output')
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
filename_ts = f'trajectories_lines_filtered_20km_{timestamp}.json'
filename_latest = 'trajectories_lines_filtered_20km_latest.json'
filtered_json_path_ts = os.path.join(output_dir, filename_ts)
filtered_json_path_latest = os.path.join(output_dir, filename_latest)

with open(filtered_json_path_ts, 'w', encoding='utf-8') as f:
    json.dump(short_trajectories, f, ensure_ascii=False, indent=2)

# Also save/overwrite a stable "latest" file for easy loading
with open(filtered_json_path_latest, 'w', encoding='utf-8') as f:
    json.dump(short_trajectories, f, ensure_ascii=False, indent=2)

print(f"\nSaved filtered trajectories to: {filtered_json_path_ts}")
print(f"Also wrote latest copy: {filtered_json_path_latest}")
print(f"Features in filtered file: {len(short_features)}")
print(f"calc_time (UTC): {calc_time}")

Computing lengths for 200 features...
  LineStrings < 20km: 114

Saved filtered trajectories to: c:\Users\adrie\github\4040_GP2\Hackathon\BuildABridge\Visualisierung\..\Visualisierung\output_vis\trajectories_lines_filtered_20km.json
Features in filtered file: 114

Computation complete!
  Features processed: 200
  LineStrings with lengths: 200
  min length (km): 0.00
  max length (km): 189.53
  mean length (km): 29.58


### Create new JSON with Distance in km

In [194]:
import json

# Load the filtered trajectories from the newly created file
filtered_json_path = '../Visualisierung/output_vis/trajectories_lines_filtered_20km.json'
with open(filtered_json_path, 'r') as f:
    filtered_trajectories = json.load(f)

# Add the filtered GeoJSON data to the map
map_1.add_data(data=filtered_trajectories, name='Trajectories')
print(f"Added {len(filtered_trajectories.get('features', []))} filtered features to map")

Added 114 filtered features to map


In [195]:
# Save map as HTML file
import os

# Define output folder (output_vis in the current Visualisierung directory)
output_dir = os.path.join(os.getcwd(), 'output_vis')

# Create folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save HTML file
html_path = os.path.join(output_dir, 'trajectories_map.html')

map_1.save_to_html(file_name=html_path)

Map saved to c:\Users\adrie\github\4040_GP2\Hackathon\BuildABridge\Visualisierung\output_vis\trajectories_map.html!


### Calculate Drivingtime

In [207]:
speed = 15  # Average speed in km/h

time = []

for feature_time in trajectories.get('features', []):
    feature_time.setdefault('properties', {})['time (min)']

    cal_time = length_20 / speed * 60  # Time in minutes
    feature_time['properties']['time (min)'] = round(cal_time, 2)
    time.append(feature_time.get('properties', {}).get('time (min)'))

print(length_20)
print(time)


TypeError: unsupported operand type(s) for /: 'list' and 'int'